In [ ]:
# Mount Google Drive and initialize Naive Bayes experiment paths

from google.colab import drive
from pathlib import Path

import json
import random
import time
import hashlib
import joblib
import platform
import sys

import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.naive_bayes import MultinomialNB
from sklearn.multiclass import OneVsRestClassifier

drive.mount('/content/drive')

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

CONFIG_DIR = PROJECT_ROOT / '00_config'
PROCESSED_DIR = PROJECT_ROOT / '01_data' / 'processed'
SPLITS_DIR = PROJECT_ROOT / '01_data' / 'splits'
FEATURES_DIR = PROJECT_ROOT / '01_data' / 'features'
SRC_DIR = PROJECT_ROOT / '03_src'

CLASSICAL_DIR = PROJECT_ROOT / '02_notebooks' / '03_classical_ml'
NOTEBOOK_DIR = CLASSICAL_DIR / '01_naive_bayes'

RESULTS_DIR = NOTEBOOK_DIR / 'results'
TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'
LOGS_DIR = RESULTS_DIR / 'logs'
RUN_CONFIGS_DIR = RESULTS_DIR / 'configs'

MODELS_DIR = (
    PROJECT_ROOT / '04_models' / 'classical' / 'naive_bayes'
)

REGISTRY_PATH = CONFIG_DIR / 'experiment_registry.csv'

assert NOTEBOOK_DIR.is_dir(), "Notebook directory not found"

for folder in [
    TABLES_DIR,
    FIGURES_DIR,
    PREDICTIONS_DIR,
    LOGS_DIR,
    RUN_CONFIGS_DIR,
    MODELS_DIR,
    SRC_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# Load shared project configuration

with open(CONFIG_DIR / 'project_config.json', 'r') as file:
    config = json.load(file)

LABELS = config['labels']
SEED = config['seed']

random.seed(SEED)
np.random.seed(SEED)

MODEL_NAME = 'Multinomial Naive Bayes'

FEATURE_REPRESENTATION = (
    config['classical_features']['baseline_representation']
)

FEATURE_BUDGET = (
    config['classical_features']['feature_budget']
)

assert config['classical_features']['status'] == 'verified'
assert REGISTRY_PATH.is_file()

sys.path.insert(0, str(SRC_DIR))

# Configure consistent visualization settings

sns.set_theme(style='whitegrid')

LABEL_COLORS = {
    'toxic': '#4C78A8',
    'severe_toxic': '#F58518',
    'obscene': '#54A24B',
    'threat': '#E45756',
    'insult': '#B279A2',
    'identity_hate': '#72B7B2'
}

# Record experiment environment versions

environment_df = pd.DataFrame([
    {'Package': 'Python', 'Version': platform.python_version()},
    {'Package': 'NumPy', 'Version': np.__version__},
    {'Package': 'Pandas', 'Version': pd.__version__},
    {'Package': 'SciPy', 'Version': scipy.__version__},
    {'Package': 'Scikit-learn', 'Version': sklearn.__version__},
    {'Package': 'Joblib', 'Version': joblib.__version__}
])

environment_df.to_csv(
    TABLES_DIR / 'environment_versions.csv',
    index=False
)

display(environment_df)

print("\nNotebook:", NOTEBOOK_DIR.name)
print("Model:", MODEL_NAME)
print("Feature representation:", FEATURE_REPRESENTATION)
print("Feature budget:", FEATURE_BUDGET)
print("Target labels:", LABELS)
print("Random seed:", SEED)
print("Results directory:", RESULTS_DIR)
print("Model directory:", MODELS_DIR)
print("Shared source directory:", SRC_DIR)
print("Experiment registry:", REGISTRY_PATH)

print("\nNaive Bayes environment initialized successfully.")

In [ ]:
# Load and verify the saved word TF-IDF matrices and source datasets

dataset_manifest = pd.read_csv(
    SPLITS_DIR / 'prepared_dataset_manifest.csv'
)

feature_manifest = pd.read_csv(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
)

def calculate_sha256(path):
    sha256 = hashlib.sha256()

    with open(path, 'rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            sha256.update(chunk)

    return sha256.hexdigest()


loaded_partitions = {}
verification_records = []

for partition, source_partition in [
    ('train', 'train_representative'),
    ('validation', 'validation')
]:

    # Verify the source dataset fingerprint
    dataset_record = dataset_manifest.loc[
        dataset_manifest['Partition'] == source_partition
    ].iloc[0]

    dataset_path = PROJECT_ROOT / dataset_record['Dataset Path']
    dataset_hash = calculate_sha256(dataset_path)

    assert dataset_hash == dataset_record['SHA256']

    df = pd.read_parquet(dataset_path)

    assert len(df) == dataset_record['Rows']
    assert df['id'].is_unique
    assert df[LABELS].isin([0, 1]).all().all()

    # Verify the saved feature matrix fingerprint
    feature_record = feature_manifest.loc[
        (feature_manifest['Artifact Type'] == 'Sparse Matrix')
        & (feature_manifest['Representation'] == FEATURE_REPRESENTATION)
        & (feature_manifest['Partition'] == partition)
    ].iloc[0]

    feature_path = PROJECT_ROOT / feature_record['Path']

    assert calculate_sha256(feature_path) == feature_record['SHA256']
    assert feature_record['Source Dataset SHA256'] == dataset_hash

    matrix = sp.load_npz(feature_path)

    assert matrix.format == 'csr'
    assert matrix.dtype == np.float32
    assert matrix.shape == (len(df), FEATURE_BUDGET)
    assert np.isfinite(matrix.data).all()
    assert (matrix.data >= 0).all()

    # Verify the feature row manifest
    row_record = feature_manifest.loc[
        (feature_manifest['Artifact Type'] == 'Row Manifest')
        & (feature_manifest['Partition'] == partition)
    ].iloc[0]

    row_path = PROJECT_ROOT / row_record['Path']

    assert calculate_sha256(row_path) == row_record['SHA256']
    assert row_record['Source Dataset SHA256'] == dataset_hash

    row_manifest = pd.read_csv(
        row_path,
        dtype={'id': str}
    )

    assert row_manifest['id'].tolist() == df['id'].tolist()
    assert row_manifest['group_id'].tolist() == df['group_id'].tolist()

    assert np.array_equal(
        row_manifest['row_index'].to_numpy(),
        np.arange(len(df))
    )

    labels = df[LABELS].to_numpy(dtype=np.int8)

    assert labels.shape == (matrix.shape[0], len(LABELS))

    loaded_partitions[partition] = {
        'data': df,
        'features': matrix,
        'labels': labels
    }

    verification_records.append({
        'Partition': partition,
        'Comments': matrix.shape[0],
        'Features': matrix.shape[1],
        'Nonzero Elements': matrix.nnz,
        'Data Type': str(matrix.dtype),
        'Dataset Hash Verified': True,
        'Feature Hash Verified': True,
        'Row Manifest Verified': True,
        'Row Alignment Verified': True,
        'Nonnegative Features': True,
        'Valid Labels': True
    })

    print(f"{partition}: Dataset and features verified")

# Assign model inputs and target labels

train_df = loaded_partitions['train']['data']
val_df = loaded_partitions['validation']['data']

X_train = loaded_partitions['train']['features']
X_val = loaded_partitions['validation']['features']

y_train = loaded_partitions['train']['labels']
y_val = loaded_partitions['validation']['labels']

assert set(train_df['id']).isdisjoint(val_df['id'])
assert set(train_df['group_id']).isdisjoint(val_df['group_id'])

# Save verification results

verification_df = pd.DataFrame(verification_records)

display(verification_df)

verification_df.to_csv(
    TABLES_DIR / 'dataset_feature_verification.csv',
    index=False
)

# Calculate label support and class imbalance

label_support_df = pd.DataFrame({
    'Label': LABELS,
    'Training Positives': y_train.sum(axis=0),
    'Training Negatives': len(y_train) - y_train.sum(axis=0),
    'Validation Positives': y_val.sum(axis=0),
    'Validation Negatives': len(y_val) - y_val.sum(axis=0),
    'Training Prevalence (%)': y_train.mean(axis=0) * 100
})

label_support_df['Training Imbalance Ratio'] = (
    label_support_df['Training Negatives']
    / label_support_df['Training Positives']
)

display(label_support_df.round(4))

label_support_df.to_csv(
    TABLES_DIR / 'label_support_and_imbalance.csv',
    index=False
)

print("\nTraining feature shape:", X_train.shape)
print("Validation feature shape:", X_val.shape)
print("Training label shape:", y_train.shape)
print("Validation label shape:", y_val.shape)

print("Feature representation:", FEATURE_REPRESENTATION)
print("Training-validation ID overlap: 0")
print("Training-validation group overlap: 0")
print("Official test dataset: Not loaded")

print("\nDataset and feature verification completed successfully.")

In [ ]:
# Create shared evaluation functions for classical and transformer models

import importlib

evaluation_code = '''
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_fscore_support,
    precision_recall_curve,
    f1_score,
    hamming_loss,
    accuracy_score
)


def validate_inputs(y_true, y_scores, labels):

    y_true = np.asarray(y_true)
    y_scores = np.asarray(y_scores, dtype=np.float64)

    if y_true.ndim != 2 or y_true.shape != y_scores.shape:
        raise ValueError("Invalid target or score dimensions")

    if y_true.shape[1] != len(labels):
        raise ValueError("Label count does not match input dimensions")

    if len(set(labels)) != len(labels):
        raise ValueError("Label names must be unique")

    if not np.isin(y_true, [0, 1]).all():
        raise ValueError("Targets must contain binary labels")

    if not np.isfinite(y_scores).all():
        raise ValueError("Prediction scores contain invalid values")

    if any(np.unique(y_true[:, i]).size != 2
           for i in range(len(labels))):
        raise ValueError("Each label must contain both classes")

    return y_true.astype(np.int8), y_scores


def apply_thresholds(y_scores, thresholds):

    y_scores = np.asarray(y_scores, dtype=np.float64)
    thresholds = np.asarray(thresholds, dtype=np.float64)

    if y_scores.ndim != 2:
        raise ValueError("Prediction scores must be two-dimensional")

    if thresholds.ndim == 0:
        thresholds = np.full(y_scores.shape[1], thresholds.item())

    if thresholds.shape != (y_scores.shape[1],):
        raise ValueError("Invalid threshold dimensions")

    if not np.isfinite(thresholds).all():
        raise ValueError("Thresholds contain invalid values")

    return (y_scores >= thresholds).astype(np.int8)


def evaluate_multilabel(y_true, y_scores, labels, thresholds=0.5):

    y_true, y_scores = validate_inputs(
        y_true, y_scores, labels
    )

    y_pred = apply_thresholds(y_scores, thresholds)

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average=None,
            zero_division=0
        )
    )

    ap_scores = average_precision_score(
        y_true, y_scores, average=None
    )

    auc_scores = roc_auc_score(
        y_true, y_scores, average=None
    )

    tp = ((y_true == 1) & (y_pred == 1)).sum(axis=0)
    tn = ((y_true == 0) & (y_pred == 0)).sum(axis=0)
    fp = ((y_true == 0) & (y_pred == 1)).sum(axis=0)
    fn = ((y_true == 1) & (y_pred == 0)).sum(axis=0)

    threshold_values = np.asarray(thresholds)

    if threshold_values.ndim == 0:
        threshold_values = np.full(
            len(labels), threshold_values.item()
        )

    per_label = pd.DataFrame({
        "Label": labels,
        "Threshold": threshold_values,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Average Precision": ap_scores,
        "ROC-AUC": auc_scores,
        "Positive Support": support,
        "Negative Support": len(y_true) - support,
        "Predicted Positives": y_pred.sum(axis=0),
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn
    })

    overall = {
        "Macro AP": float(np.mean(ap_scores)),
        "Micro AP": float(average_precision_score(
            y_true, y_scores, average="micro"
        )),
        "Macro ROC-AUC": float(np.mean(auc_scores)),
        "Micro ROC-AUC": float(roc_auc_score(
            y_true, y_scores, average="micro"
        )),
        "Macro Precision": float(np.mean(precision)),
        "Macro Recall": float(np.mean(recall)),
        "Macro F1": float(np.mean(f1)),
        "Micro F1": float(f1_score(
            y_true, y_pred, average="micro", zero_division=0
        )),
        "Hamming Loss": float(hamming_loss(y_true, y_pred)),
        "Exact Match Accuracy": float(
            accuracy_score(y_true, y_pred)
        )
    }

    return overall, per_label


def optimize_thresholds(y_true, y_scores, labels):

    y_true, y_scores = validate_inputs(
        y_true, y_scores, labels
    )

    records = []

    for i, label in enumerate(labels):

        precision, recall, thresholds = precision_recall_curve(
            y_true[:, i],
            y_scores[:, i]
        )

        # The final precision-recall point has no threshold
        precision = precision[:-1]
        recall = recall[:-1]

        denominator = precision + recall

        f1 = np.divide(
            2 * precision * recall,
            denominator,
            out=np.zeros_like(denominator),
            where=denominator > 0
        )

        # Prefer the highest threshold when F1 scores are tied
        best_indices = np.flatnonzero(
            np.isclose(
                f1, f1.max(),
                rtol=0, atol=1e-12
            )
        )

        best_idx = best_indices[-1]

        records.append({
            "Label": label,
            "Optimal Threshold": float(thresholds[best_idx]),
            "Validation Precision": float(precision[best_idx]),
            "Validation Recall": float(recall[best_idx]),
            "Validation F1": float(f1[best_idx])
        })

    return pd.DataFrame(records)
'''

module_path = SRC_DIR / 'evaluation_utils.py'

# Avoid silently changing an existing shared evaluation module

if module_path.exists():
    assert module_path.read_text(encoding='utf-8') == evaluation_code, \
        "Existing evaluation module differs from this version"
else:
    module_path.write_text(evaluation_code, encoding='utf-8')

importlib.invalidate_caches()

import evaluation_utils
importlib.reload(evaluation_utils)

from evaluation_utils import (
    apply_thresholds,
    evaluate_multilabel,
    optimize_thresholds
)

print("Evaluation module saved:", module_path)

# Verify evaluation functions using a small synthetic dataset

sample_labels = ['label_1', 'label_2']

sample_targets = np.array([
    [1, 0],
    [0, 1],
    [1, 1],
    [0, 0]
])

sample_scores = np.array([
    [0.80, 0.10],
    [0.20, 0.45],
    [0.70, 0.40],
    [0.10, 0.20]
])

default_metrics, _ = evaluate_multilabel(
    sample_targets,
    sample_scores,
    sample_labels,
    thresholds=0.5
)

sample_thresholds = optimize_thresholds(
    sample_targets,
    sample_scores,
    sample_labels
)

optimized_metrics, _ = evaluate_multilabel(
    sample_targets,
    sample_scores,
    sample_labels,
    thresholds=sample_thresholds['Optimal Threshold'].to_numpy()
)

# Verify metric calculations and threshold behavior

assert np.isclose(default_metrics['Macro F1'], 0.5)
assert np.isclose(optimized_metrics['Macro F1'], 1.0)
assert np.isclose(optimized_metrics['Micro F1'], 1.0)
assert np.isclose(optimized_metrics['Macro AP'], 1.0)
assert np.isclose(optimized_metrics['Macro ROC-AUC'], 1.0)
assert np.isclose(optimized_metrics['Hamming Loss'], 0.0)
assert np.isclose(optimized_metrics['Exact Match Accuracy'], 1.0)

assert np.isclose(
    sample_thresholds.loc[0, 'Optimal Threshold'], 0.70
)

assert np.isclose(
    sample_thresholds.loc[1, 'Optimal Threshold'], 0.40
)

# Verify compatibility with signed SVM decision margins

margin_scores = sample_scores - 0.5

margin_thresholds = optimize_thresholds(
    sample_targets,
    margin_scores,
    sample_labels
)

margin_metrics, _ = evaluate_multilabel(
    sample_targets,
    margin_scores,
    sample_labels,
    thresholds=margin_thresholds['Optimal Threshold'].to_numpy()
)

assert np.isclose(margin_metrics['Macro F1'], 1.0)

verification_df = pd.DataFrame([
    {
        'Check': 'Default threshold evaluation',
        'Status': 'Passed'
    },
    {
        'Check': 'Per-label threshold optimization',
        'Status': 'Passed'
    },
    {
        'Check': 'Ranking metrics',
        'Status': 'Passed'
    },
    {
        'Check': 'Multi-label error metrics',
        'Status': 'Passed'
    },
    {
        'Check': 'Signed decision margin compatibility',
        'Status': 'Passed'
    }
])

verification_df.to_csv(
    TABLES_DIR / 'evaluation_module_verification.csv',
    index=False
)

print("\\nSynthetic threshold optimization results")
display(sample_thresholds.round(4))

print("\\nEvaluation module verification")
display(verification_df)

print("Default Macro F1:", default_metrics['Macro F1'])
print("Optimized Macro F1:", optimized_metrics['Macro F1'])
print("Module SHA256:", calculate_sha256(module_path))

print("\\nShared evaluation framework initialized successfully.")

In [ ]:
# Define the Multinomial Naive Bayes baseline configuration

RUN_ID = 'NB_WORD_BASELINE_S42'

NB_PARAMS = {
    'alpha': 1.0,
    'force_alpha': True,
    'fit_prior': True,
    'class_prior': None
}

OVR_PARAMS = {
    'n_jobs': 1
}

DEFAULT_THRESHOLD = 0.5

# Initialize six independent binary classifiers through One-vs-Rest

nb_model = OneVsRestClassifier(
    estimator=MultinomialNB(**NB_PARAMS),
    **OVR_PARAMS
)

# Define experiment configuration

run_config = {
    'run_id': RUN_ID,
    'notebook': '03_classical_ml/01_naive_bayes',
    'model': MODEL_NAME,
    'classification_strategy': 'one_vs_rest',
    'feature_representation': FEATURE_REPRESENTATION,
    'feature_count': int(X_train.shape[1]),
    'sampling_strategy': 'representative',
    'training_samples': len(y_train),
    'validation_samples': len(y_val),
    'labels': LABELS,
    'num_labels': len(LABELS),
    'seed': SEED,
    'model_parameters': NB_PARAMS,
    'wrapper_parameters': OVR_PARAMS,
    'class_imbalance_handling': 'none',
    'score_type': 'positive_class_probability',
    'default_threshold': DEFAULT_THRESHOLD,
    'primary_selection_metric': 'macro_average_precision',
    'threshold_optimization_metric': 'per_label_f1',
    'threshold_tie_breaking': 'highest_threshold',
    'evaluation_module_sha256': calculate_sha256(
        SRC_DIR / 'evaluation_utils.py'
    ),
    'training_dataset_sha256': dataset_manifest.loc[
        dataset_manifest['Partition'] == 'train_representative',
        'SHA256'
    ].iloc[0],
    'validation_dataset_sha256': dataset_manifest.loc[
        dataset_manifest['Partition'] == 'validation',
        'SHA256'
    ].iloc[0],
    'feature_manifest_sha256': calculate_sha256(
        FEATURES_DIR / 'feature_artifact_manifest.csv'
    ),
    'sklearn_version': sklearn.__version__,
    'official_test_used': False
}

RUN_CONFIG_PATH = RUN_CONFIGS_DIR / f'{RUN_ID}.json'
MODEL_PATH = MODELS_DIR / f'{RUN_ID}.joblib'

with open(RUN_CONFIG_PATH, 'w') as file:
    json.dump(run_config, file, indent=4)

# Calculate expected empirical class priors

positive_priors = y_train.mean(axis=0)

class_prior_df = pd.DataFrame({
    'Label': LABELS,
    'Training Positives': y_train.sum(axis=0),
    'Training Negatives': len(y_train) - y_train.sum(axis=0),
    'Expected Positive Prior': positive_priors,
    'Expected Negative Prior': 1 - positive_priors,
    'Positive Prior (%)': positive_priors * 100
})

display(class_prior_df.round(6))

class_prior_df.to_csv(
    TABLES_DIR / 'expected_class_priors.csv',
    index=False
)

print("\nRun ID:", RUN_ID)
print("Model:", MODEL_NAME)
print("Classification strategy: One-vs-Rest")
print("Binary classifiers:", len(LABELS))
print("Smoothing alpha:", NB_PARAMS['alpha'])
print("Learn empirical class priors:", NB_PARAMS['fit_prior'])
print("Class imbalance handling: None")
print("Default probability threshold:", DEFAULT_THRESHOLD)
print("Primary metric: Macro Average Precision")
print("Configuration saved:", RUN_CONFIG_PATH)
print("Model trained: No")

print("\nNaive Bayes configuration initialized successfully.")

In [ ]:
# Train the Multinomial Naive Bayes baseline

from datetime import datetime, timezone

training_start = datetime.now(timezone.utc).isoformat()

start_time = time.perf_counter()

nb_model.fit(X_train, y_train)

training_time = time.perf_counter() - start_time

training_end = datetime.now(timezone.utc).isoformat()

# Verify the fitted model

assert nb_model.multilabel_
assert len(nb_model.estimators_) == len(LABELS)

assert nb_model.n_features_in_ == X_train.shape[1]

training_records = []

for i, (label, estimator) in enumerate(
    zip(LABELS, nb_model.estimators_)
):

    assert np.array_equal(estimator.classes_, [0, 1])

    learned_priors = np.exp(estimator.class_log_prior_)

    expected_positive_prior = y_train[:, i].mean()

    expected_negative_prior = 1 - expected_positive_prior

    assert np.allclose(
        learned_priors,
        [expected_negative_prior, expected_positive_prior]
    )

    assert np.isfinite(estimator.feature_log_prob_).all()

    assert estimator.feature_log_prob_.shape == (
        2, X_train.shape[1]
    )

    assert np.array_equal(
        estimator.class_count_,
        np.bincount(y_train[:, i], minlength=2)
    )

    training_records.append({
        'Label': label,
        'Training Samples': int(estimator.class_count_.sum()),
        'Negative Samples': int(estimator.class_count_[0]),
        'Positive Samples': int(estimator.class_count_[1]),
        'Expected Positive Prior': expected_positive_prior,
        'Learned Positive Prior': learned_priors[1],
        'Learned Negative Prior': learned_priors[0],
        'Prior Absolute Error': abs(
            learned_priors[1] - expected_positive_prior
        ),
        'Features': estimator.n_features_in_,
        'Alpha': estimator.alpha,
        'Training Status': 'Completed'
    })

nb_training_df = pd.DataFrame(training_records)

print("Fitted binary classifier verification")
display(nb_training_df.round(8))

nb_training_df.to_csv(
    TABLES_DIR / 'nb_classifier_training_summary.csv',
    index=False
)

# Save training execution information

training_log = {
    'run_id': RUN_ID,
    'model': MODEL_NAME,
    'training_samples': len(y_train),
    'features': X_train.shape[1],
    'binary_classifiers': len(nb_model.estimators_),
    'training_start': training_start,
    'training_end': training_end,
    'training_time_seconds': training_time,
    'maximum_prior_absolute_error': float(
        nb_training_df['Prior Absolute Error'].max()
    ),
    'status': 'trained',
    'official_test_used': False
}

with open(
    LOGS_DIR / f'{RUN_ID}_training_log.json', 'w'
) as file:
    json.dump(training_log, file, indent=4)

print("\nTraining completion summary")
print("Run ID:", RUN_ID)
print("Model:", MODEL_NAME)
print("Training samples:", len(y_train))
print("Input features:", nb_model.n_features_in_)
print("Binary classifiers trained:", len(nb_model.estimators_))
print("Training time (seconds):", round(training_time, 4))
print("Multilabel classification:", nb_model.multilabel_)
print("Class prior verification: Passed")
print("Feature log probability verification: Passed")
print("Official test dataset: Not used")
print("\nMultinomial Naive Bayes training completed successfully.")

In [ ]:
# Generate positive-class probability scores on the validation dataset

start_time = time.perf_counter()

val_scores = nb_model.predict_proba(X_val)

prediction_time = time.perf_counter() - start_time

# Verify probability dimensions and numerical validity

assert val_scores.shape == (len(y_val), len(LABELS))
assert np.isfinite(val_scores).all()
assert ((val_scores >= 0) & (val_scores <= 1)).all()

assert np.array_equal(
    nb_model.classes_,
    np.arange(len(LABELS))
)

# Verify probability column alignment

for i, estimator in enumerate(nb_model.estimators_):

    individual_scores = estimator.predict_proba(X_val[:100])[:, 1]

    assert np.allclose(
        val_scores[:100, i],
        individual_scores
    )

# Summarize the validation probability distributions

score_records = []

for i, label in enumerate(LABELS):

    scores = val_scores[:, i]

    positive_scores = scores[y_val[:, i] == 1]
    negative_scores = scores[y_val[:, i] == 0]

    score_records.append({
        'Label': label,
        'Minimum Score': scores.min(),
        'Maximum Score': scores.max(),
        'Mean Score': scores.mean(),
        'Median Score': np.median(scores),
        'Mean Positive Score': positive_scores.mean(),
        'Mean Negative Score': negative_scores.mean(),
        'Median Positive Score': np.median(positive_scores),
        'Median Negative Score': np.median(negative_scores),
        '95th Percentile': np.quantile(scores, 0.95),
        '99th Percentile': np.quantile(scores, 0.99),
        'Actual Positives': len(positive_scores),
        'Scores Above Default Threshold':
            (scores >= DEFAULT_THRESHOLD).sum()
    })

score_summary_df = pd.DataFrame(score_records)

print("Validation probability score summary")
display(score_summary_df.round(6))

score_summary_df.to_csv(
    TABLES_DIR / 'nb_validation_score_summary.csv',
    index=False
)

# Save raw validation probabilities and corresponding ground truth

validation_predictions_df = pd.DataFrame({
    'id': val_df['id'].to_numpy()
})

for i, label in enumerate(LABELS):

    validation_predictions_df[f'{label}_true'] = y_val[:, i]
    validation_predictions_df[f'{label}_score'] = val_scores[:, i]

validation_predictions_df.to_parquet(
    PREDICTIONS_DIR / f'{RUN_ID}_validation_scores.parquet',
    index=False
)

# Compare scores for actual positive and negative comments

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

PLOT_SCORE_FLOOR = 1e-15

for i, (ax, label) in enumerate(zip(axes.flat, LABELS)):

    for target, name, color in [
        (0, 'Actual negative', '#4C78A8'),
        (1, 'Actual positive', '#E45756')
    ]:

        scores = val_scores[y_val[:, i] == target, i]

        # Clip only for logarithmic visualization
        sorted_scores = np.sort(
            np.clip(scores, PLOT_SCORE_FLOOR, 1.0)
        )

        cumulative = (
            np.arange(1, len(sorted_scores) + 1)
            / len(sorted_scores)
        )

        ax.step(
            sorted_scores,
            cumulative,
            where='post',
            color=color,
            linewidth=2,
            label=f'{name} (n={len(scores):,})'
        )

    ax.axvline(
        DEFAULT_THRESHOLD,
        color='#54A24B',
        linestyle='--',
        linewidth=1.5,
        label='Default threshold'
    )

    ax.set_xscale('log')
    ax.set_xlim(PLOT_SCORE_FLOOR, 1)
    ax.set_ylim(0, 1.05)

    ax.set_title(
        label.replace('_', ' ').title(),
        fontsize=13
    )

    ax.set_xlabel('Predicted Positive-Class Probability')
    ax.set_ylabel('Cumulative Proportion')
    ax.legend(fontsize=9)

fig.suptitle(
    'Naive Bayes Validation Probability Distributions',
    fontsize=17
)

fig.text(
    0.5, 0.01,
    'Logarithmic probability axis. Scores below 1e-15 are '
    'clipped for visualization only.',
    ha='center',
    fontsize=10
)

sns.despine()
plt.tight_layout(rect=[0, 0.04, 1, 0.95])

plt.savefig(
    FIGURES_DIR / 'nb_validation_score_distributions.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

print("\nValidation prediction summary")
print("Validation comments:", len(y_val))
print("Probability matrix shape:", val_scores.shape)
print("Total probability scores:", val_scores.size)
print("Prediction time (seconds):", round(prediction_time, 4))
print("Probability column alignment: Verified")
print("Raw validation scores saved successfully.")
print("\nValidation probability analysis completed.")

In [ ]:
# Evaluate Naive Bayes using the default probability threshold

default_metrics, default_per_label_df = evaluate_multilabel(
    y_true=y_val,
    y_scores=val_scores,
    labels=LABELS,
    thresholds=DEFAULT_THRESHOLD
)

default_predictions = apply_thresholds(
    val_scores,
    DEFAULT_THRESHOLD
)

# Verify confusion counts and positive predictions

assert default_predictions.shape == y_val.shape

assert np.array_equal(
    default_per_label_df['Positive Support'].to_numpy(),
    y_val.sum(axis=0)
)

assert np.array_equal(
    default_per_label_df['Predicted Positives'].to_numpy(),
    default_predictions.sum(axis=0)
)

assert (
    default_per_label_df[['TP', 'TN', 'FP', 'FN']]
    .sum(axis=1).eq(len(y_val)).all()
)

assert (
    default_per_label_df['TP'] + default_per_label_df['FN']
    == default_per_label_df['Positive Support']
).all()

# Construct the overall performance table

default_overall_df = pd.DataFrame([{
    'Run ID': RUN_ID,
    'Model': MODEL_NAME,
    'Feature Representation': FEATURE_REPRESENTATION,
    'Partition': 'validation',
    'Threshold Strategy': 'default',
    **default_metrics
}])

print("Overall validation performance - default threshold")

display(
    pd.DataFrame(
        list(default_metrics.items()),
        columns=['Metric', 'Value']
    ).round(6)
)

print("\nPer-label validation performance")

display(default_per_label_df.round(6))

# Identify labels with no positive predictions

zero_prediction_labels = default_per_label_df.loc[
    default_per_label_df['Predicted Positives'] == 0,
    'Label'
].tolist()

print("\nLabels with zero positive predictions:")
print(zero_prediction_labels)

# Calculate the no-skill reference for Average Precision

ap_reference_df = default_per_label_df[[
    'Label',
    'Positive Support',
    'Average Precision',
    'ROC-AUC'
]].copy()

ap_reference_df['No-Skill AP Reference'] = (
    ap_reference_df['Positive Support'] / len(y_val)
)

ap_reference_df['AP Lift'] = (
    ap_reference_df['Average Precision']
    / ap_reference_df['No-Skill AP Reference']
)

print("\nRanking performance against no-skill reference")
display(ap_reference_df.round(6))

# Compare against an all-negative classification baseline

all_negative_predictions = np.zeros_like(y_val)

all_negative_scores = np.zeros_like(
    y_val, dtype=np.float64
)

all_negative_metrics, _ = evaluate_multilabel(
    y_true=y_val,
    y_scores=all_negative_scores,
    labels=LABELS,
    thresholds=0.5
)

baseline_comparison_df = pd.DataFrame([
    {
        'Model': 'All-Negative Baseline',
        **all_negative_metrics
    },
    {
        'Model': MODEL_NAME,
        **default_metrics
    }
])

print("\nComparison with all-negative baseline")

display(baseline_comparison_df.round(6))

# Save default threshold evaluation results

default_overall_df.to_csv(
    TABLES_DIR / 'nb_default_overall_metrics.csv',
    index=False
)

default_per_label_df.to_csv(
    TABLES_DIR / 'nb_default_per_label_metrics.csv',
    index=False
)

ap_reference_df.to_csv(
    TABLES_DIR / 'nb_average_precision_reference.csv',
    index=False
)

baseline_comparison_df.to_csv(
    TABLES_DIR / 'nb_all_negative_baseline_comparison.csv',
    index=False
)

print("\nDefault threshold evaluation summary")
print("Macro AP:", round(default_metrics['Macro AP'], 6))
print("Macro F1:", round(default_metrics['Macro F1'], 6))
print("Micro F1:", round(default_metrics['Micro F1'], 6))
print("Macro ROC-AUC:",
      round(default_metrics['Macro ROC-AUC'], 6))
print("Hamming Loss:",
      round(default_metrics['Hamming Loss'], 6))
print("Exact Match Accuracy:",
      round(default_metrics['Exact Match Accuracy'], 6))

print("Labels with zero positive predictions:",
      len(zero_prediction_labels))

print("Official test dataset: Not used")
print("\nDefault threshold evaluation completed successfully.")

In [ ]:
# Optimize individual label thresholds using validation predictions

threshold_df = optimize_thresholds(
    y_true=y_val,
    y_scores=val_scores,
    labels=LABELS
)

assert threshold_df['Label'].tolist() == LABELS

optimized_thresholds = threshold_df[
    'Optimal Threshold'
].to_numpy(dtype=np.float64)

assert np.isfinite(optimized_thresholds).all()
assert ((optimized_thresholds >= 0) &
        (optimized_thresholds <= 1)).all()

# Evaluate predictions using optimized thresholds

optimized_metrics, optimized_per_label_df = evaluate_multilabel(
    y_true=y_val,
    y_scores=val_scores,
    labels=LABELS,
    thresholds=optimized_thresholds
)

optimized_predictions = apply_thresholds(
    val_scores,
    optimized_thresholds
)

# Verify threshold optimization results

assert optimized_predictions.shape == y_val.shape

assert np.allclose(
    threshold_df['Validation F1'],
    optimized_per_label_df['F1']
)

assert np.all(
    optimized_per_label_df['F1'].to_numpy()
    >= default_per_label_df['F1'].to_numpy() - 1e-10
)

# Ranking metrics must remain unchanged after threshold tuning

ranking_metrics = [
    'Macro AP',
    'Micro AP',
    'Macro ROC-AUC',
    'Micro ROC-AUC'
]

for metric in ranking_metrics:

    assert np.isclose(
        default_metrics[metric],
        optimized_metrics[metric]
    )

# Compare default and optimized thresholds

threshold_comparison_df = pd.DataFrame({
    'Label': LABELS,
    'Default Threshold': DEFAULT_THRESHOLD,
    'Optimized Threshold': optimized_thresholds,
    'Default Precision': default_per_label_df['Precision'],
    'Optimized Precision': optimized_per_label_df['Precision'],
    'Default Recall': default_per_label_df['Recall'],
    'Optimized Recall': optimized_per_label_df['Recall'],
    'Default F1': default_per_label_df['F1'],
    'Optimized F1': optimized_per_label_df['F1'],
    'Default Predicted Positives':
        default_per_label_df['Predicted Positives'],
    'Optimized Predicted Positives':
        optimized_per_label_df['Predicted Positives'],
    'Actual Positives':
        optimized_per_label_df['Positive Support']
})

threshold_comparison_df['F1 Improvement'] = (
    threshold_comparison_df['Optimized F1']
    - threshold_comparison_df['Default F1']
)

print("Per-label threshold optimization results")

display(threshold_comparison_df.round(8))

# Compare overall performance before and after threshold tuning

overall_comparison_df = pd.DataFrame([
    {
        'Metric': metric,
        'Default': default_metrics[metric],
        'Optimized': optimized_metrics[metric],
        'Change': optimized_metrics[metric] - default_metrics[metric]
    }
    for metric in default_metrics
])

print("\nOverall performance comparison")

display(overall_comparison_df.round(6))

print("\nOptimized per-label performance")

display(optimized_per_label_df.round(6))

# Save threshold optimization results

threshold_df.to_csv(
    TABLES_DIR / 'nb_optimized_thresholds.csv',
    index=False
)

threshold_comparison_df.to_csv(
    TABLES_DIR / 'nb_threshold_comparison.csv',
    index=False
)

overall_comparison_df.to_csv(
    TABLES_DIR / 'nb_overall_threshold_comparison.csv',
    index=False
)

optimized_per_label_df.to_csv(
    TABLES_DIR / 'nb_optimized_per_label_metrics.csv',
    index=False
)

optimized_overall_df = pd.DataFrame([{
    'Run ID': RUN_ID,
    'Model': MODEL_NAME,
    'Feature Representation': FEATURE_REPRESENTATION,
    'Partition': 'validation',
    'Threshold Strategy': 'per_label_optimized',
    **optimized_metrics
}])

optimized_overall_df.to_csv(
    TABLES_DIR / 'nb_optimized_overall_metrics.csv',
    index=False
)

# Save full-precision thresholds for future evaluation

threshold_config = {
    'run_id': RUN_ID,
    'selection_partition': 'validation',
    'optimization_metric': 'per_label_f1',
    'comparison_operator': 'greater_than_or_equal',
    'tie_breaking': 'highest_threshold',
    'score_type': 'positive_class_probability',
    'default_threshold': DEFAULT_THRESHOLD,
    'labels': LABELS,
    'thresholds': {
        label: float(threshold)
        for label, threshold in zip(
            LABELS, optimized_thresholds
        )
    },
    'official_test_used': False
}

THRESHOLD_PATH = (
    RUN_CONFIGS_DIR / f'{RUN_ID}_thresholds.json'
)

with open(THRESHOLD_PATH, 'w') as file:
    json.dump(threshold_config, file, indent=4)

# Summarize improvements

print("\nThreshold optimization summary")

print("Default Macro F1:",
      round(default_metrics['Macro F1'], 6))

print("Optimized Macro F1:",
      round(optimized_metrics['Macro F1'], 6))

print("Macro F1 improvement:",
      round(optimized_metrics['Macro F1']
            - default_metrics['Macro F1'], 6))

print("Default Micro F1:",
      round(default_metrics['Micro F1'], 6))

print("Optimized Micro F1:",
      round(optimized_metrics['Micro F1'], 6))

print("Macro AP:", round(optimized_metrics['Macro AP'], 6))

print("Ranking metrics unchanged: Verified")

print("Labels with zero positive predictions:",
      (optimized_per_label_df['Predicted Positives'] == 0).sum())

print("Model retrained: No")
print("Official test dataset: Not used")

print("\nPer-label threshold optimization completed successfully.")

In [ ]:
# Visualize Naive Bayes validation performance and threshold effects

from sklearn.metrics import precision_recall_curve

label_names = [
    label.replace('_', ' ').title()
    for label in LABELS
]

# Compare default and optimized classification metrics

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

x = np.arange(len(LABELS))
width = 0.35

for ax, metric in zip(
    axes.flat[:3],
    ['Precision', 'Recall', 'F1']
):

    default_values = default_per_label_df[metric].to_numpy()
    optimized_values = optimized_per_label_df[metric].to_numpy()

    bars_default = ax.bar(
        x - width / 2,
        default_values,
        width,
        color='#4C78A8',
        label='Default'
    )

    bars_optimized = ax.bar(
        x + width / 2,
        optimized_values,
        width,
        color='#54A24B',
        label='Optimized'
    )

    ax.bar_label(
        bars_default, fmt='%.3f',
        padding=3, fontsize=9
    )

    ax.bar_label(
        bars_optimized, fmt='%.3f',
        padding=3, fontsize=9
    )

    ax.set_title(f'Per-Label {metric}')
    ax.set_xticks(x)
    ax.set_xticklabels(label_names, rotation=20, ha='right')
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1.15)
    ax.legend()
    ax.grid(axis='x', visible=False)

# Compare overall threshold-dependent metrics

overall_plot_metrics = [
    'Macro Precision',
    'Macro Recall',
    'Macro F1',
    'Micro F1'
]

overall_plot_df = overall_comparison_df.set_index(
    'Metric'
).loc[
    overall_plot_metrics,
    ['Default', 'Optimized']
]

overall_plot_df.plot(
    kind='bar',
    ax=axes[1, 1],
    color=['#4C78A8', '#54A24B'],
    width=0.7
)

for container in axes[1, 1].containers:
    axes[1, 1].bar_label(
        container,
        fmt='%.3f',
        padding=3,
        fontsize=9
    )

axes[1, 1].set_title('Overall Classification Performance')
axes[1, 1].set_xlabel('')
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_ylim(0, 1.05)
axes[1, 1].tick_params(axis='x', rotation=15)
axes[1, 1].grid(axis='x', visible=False)
axes[1, 1].legend()

fig.suptitle(
    'Naive Bayes: Default vs Optimized Thresholds',
    fontsize=17
)

sns.despine()
plt.tight_layout(rect=[0, 0, 1, 0.95])

plt.savefig(
    FIGURES_DIR / 'nb_threshold_performance_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Plot precision-recall curves for all toxicity labels

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

for i, (ax, label) in enumerate(zip(axes.flat, LABELS)):

    precision, recall, _ = precision_recall_curve(
        y_val[:, i],
        val_scores[:, i]
    )

    ap = default_per_label_df.loc[i, 'Average Precision']
    prevalence = y_val[:, i].mean()

    ax.step(
        recall,
        precision,
        where='post',
        color=LABEL_COLORS[label],
        linewidth=2,
        label=f'AP = {ap:.4f}'
    )

    ax.axhline(
        prevalence,
        color='gray',
        linestyle='--',
        linewidth=1.3,
        label=f'No-skill reference = {prevalence:.4f}'
    )

    # Mark default and optimized operating points

    ax.scatter(
        default_per_label_df.loc[i, 'Recall'],
        default_per_label_df.loc[i, 'Precision'],
        color='#4C78A8',
        marker='o',
        s=65,
        zorder=5,
        label='Default threshold'
    )

    ax.scatter(
        optimized_per_label_df.loc[i, 'Recall'],
        optimized_per_label_df.loc[i, 'Precision'],
        color='#E45756',
        marker='D',
        s=65,
        zorder=5,
        label='Optimized threshold'
    )

    ax.set_title(label_names[i], fontsize=13)
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.05)
    ax.legend(fontsize=8)

fig.suptitle(
    'Naive Bayes Validation Precision-Recall Curves',
    fontsize=17
)

sns.despine()
plt.tight_layout(rect=[0, 0, 1, 0.95])

plt.savefig(
    FIGURES_DIR / 'nb_precision_recall_curves.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Compare confusion matrices using both threshold strategies

fig, axes = plt.subplots(3, 4, figsize=(18, 13))

for i, label in enumerate(LABELS):

    row = i // 2
    column = (i % 2) * 2

    for offset, (strategy, metrics_df) in enumerate([
        ('Default', default_per_label_df),
        ('Optimized', optimized_per_label_df)
    ]):

        ax = axes[row, column + offset]

        result = metrics_df.iloc[i]

        counts = np.array([
            [result['TN'], result['FP']],
            [result['FN'], result['TP']]
        ], dtype=int)

        # Normalize by actual class to reveal rare-label errors

        normalized = (
            counts / counts.sum(axis=1, keepdims=True)
        )

        annotations = np.empty((2, 2), dtype=object)

        for r in range(2):
            for c in range(2):

                annotations[r, c] = (
                    f'{counts[r, c]:,}\n'
                    f'({normalized[r, c]:.1%})'
                )

        sns.heatmap(
            normalized,
            annot=annotations,
            fmt='',
            cmap='Blues',
            vmin=0,
            vmax=1,
            cbar=False,
            square=True,
            linewidths=0.5,
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'],
            ax=ax
        )

        ax.set_title(
            f'{label_names[i]} - {strategy}',
            fontsize=11
        )

        ax.set_xlabel('Predicted label')
        ax.set_ylabel('Actual label')
        ax.tick_params(axis='y', rotation=0)

fig.suptitle(
    'Naive Bayes Confusion Matrices: Threshold Comparison',
    fontsize=18
)

fig.text(
    0.5, 0.02,
    'Cells show absolute counts and percentages within each '
    'actual class. Color intensity represents row-normalized values.',
    ha='center',
    fontsize=11
)

plt.tight_layout(rect=[0, 0.05, 1, 0.95])

plt.savefig(
    FIGURES_DIR / 'nb_confusion_matrix_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

print("\nPerformance visualization completed successfully.")
print("Figures generated: 3")

In [ ]:
# Save the trained Naive Bayes model and verify reconstruction

joblib.dump(
    nb_model,
    MODEL_PATH,
    compress=3
)

reloaded_model = joblib.load(MODEL_PATH)

assert reloaded_model.multilabel_
assert len(reloaded_model.estimators_) == len(LABELS)
assert reloaded_model.n_features_in_ == X_train.shape[1]

# Verify predictions from the reloaded model

reloaded_scores = reloaded_model.predict_proba(X_val)

assert reloaded_scores.shape == val_scores.shape
assert np.allclose(
    reloaded_scores,
    val_scores,
    rtol=0,
    atol=1e-12
)

maximum_score_error = float(
    np.abs(reloaded_scores - val_scores).max()
)

# Reload full-precision thresholds

with open(THRESHOLD_PATH, 'r') as file:
    saved_threshold_config = json.load(file)

assert saved_threshold_config['labels'] == LABELS

saved_thresholds = np.array([
    saved_threshold_config['thresholds'][label]
    for label in LABELS
], dtype=np.float64)

assert np.array_equal(
    saved_thresholds,
    optimized_thresholds
)

# Verify both threshold strategies after model reload

reloaded_default_predictions = apply_thresholds(
    reloaded_scores,
    DEFAULT_THRESHOLD
)

reloaded_optimized_predictions = apply_thresholds(
    reloaded_scores,
    saved_thresholds
)

assert np.array_equal(
    reloaded_default_predictions,
    default_predictions
)

assert np.array_equal(
    reloaded_optimized_predictions,
    optimized_predictions
)

print("Model reload verification: Passed")
print("Maximum probability reconstruction error:",
      maximum_score_error)

# Save ID-aligned validation scores and predictions

final_predictions_df = pd.DataFrame({
    'id': val_df['id'].to_numpy()
})

for i, label in enumerate(LABELS):

    final_predictions_df[f'{label}_true'] = y_val[:, i]

    final_predictions_df[f'{label}_score'] = (
        val_scores[:, i]
    )

    final_predictions_df[f'{label}_pred_default'] = (
        default_predictions[:, i]
    )

    final_predictions_df[f'{label}_pred_optimized'] = (
        optimized_predictions[:, i]
    )

FINAL_PREDICTIONS_PATH = (
    PREDICTIONS_DIR
    / f'{RUN_ID}_validation_predictions.parquet'
)

final_predictions_df.to_parquet(
    FINAL_PREDICTIONS_PATH,
    index=False
)

# Verify the saved prediction file

loaded_predictions_df = pd.read_parquet(
    FINAL_PREDICTIONS_PATH
)

pd.testing.assert_frame_equal(
    final_predictions_df,
    loaded_predictions_df
)

assert loaded_predictions_df['id'].is_unique

assert (
    loaded_predictions_df['id'].tolist()
    == val_df['id'].tolist()
)

saved_targets = loaded_predictions_df[[
    f'{label}_true' for label in LABELS
]].to_numpy()

saved_scores = loaded_predictions_df[[
    f'{label}_score' for label in LABELS
]].to_numpy()

assert np.array_equal(saved_targets, y_val)
assert np.array_equal(saved_scores, val_scores)

# Recompute metrics from saved predictions

recomputed_metrics, _ = evaluate_multilabel(
    saved_targets,
    saved_scores,
    LABELS,
    thresholds=saved_thresholds
)

for metric in optimized_metrics:

    assert np.isclose(
        recomputed_metrics[metric],
        optimized_metrics[metric],
        rtol=0,
        atol=1e-12
    )

print("Saved prediction alignment: Verified")
print("Saved prediction metric reconstruction: Passed")

# Verify the original fitted word vectorizer dependency

vectorizer_record = feature_manifest.loc[
    (feature_manifest['Artifact Type'] == 'Vectorizer')
    & (
        feature_manifest['Representation']
        == FEATURE_REPRESENTATION
    )
].iloc[0]

vectorizer_path = PROJECT_ROOT / vectorizer_record['Path']

assert (
    calculate_sha256(vectorizer_path)
    == vectorizer_record['SHA256']
)

assert (
    calculate_sha256(SRC_DIR / 'evaluation_utils.py')
    == run_config['evaluation_module_sha256']
)

# Generate artifact fingerprints

artifact_paths = {
    'Trained Model': MODEL_PATH,
    'Run Configuration': RUN_CONFIG_PATH,
    'Optimized Thresholds': THRESHOLD_PATH,
    'Raw Validation Scores': (
        PREDICTIONS_DIR
        / f'{RUN_ID}_validation_scores.parquet'
    ),
    'Final Validation Predictions': FINAL_PREDICTIONS_PATH,
    'Training Log': (
        LOGS_DIR / f'{RUN_ID}_training_log.json'
    ),
    'Fitted Word Vectorizer': vectorizer_path,
    'Evaluation Module': SRC_DIR / 'evaluation_utils.py'
}

artifact_records = []

for artifact_type, path in artifact_paths.items():

    assert path.is_file()
    assert path.stat().st_size > 0

    artifact_records.append({
        'Run ID': RUN_ID,
        'Artifact Type': artifact_type,
        'Filename': path.name,
        'Path': str(path.relative_to(PROJECT_ROOT)),
        'Size Bytes': path.stat().st_size,
        'Size (MiB)': path.stat().st_size / (1024 ** 2),
        'SHA256': calculate_sha256(path),
        'Status': 'Verified'
    })

nb_artifact_manifest_df = pd.DataFrame(
    artifact_records
)

nb_artifact_manifest_df.to_csv(
    TABLES_DIR / 'nb_model_artifact_manifest.csv',
    index=False
)

print("\nModel artifact manifest")

display(
    nb_artifact_manifest_df[[
        'Artifact Type',
        'Filename',
        'Size (MiB)',
        'Status'
    ]].round(4)
)

# Save model reconstruction verification

model_reload_df = pd.DataFrame([{
    'Run ID': RUN_ID,
    'Validation Comments': len(y_val),
    'Probability Scores Verified': val_scores.size,
    'Maximum Absolute Score Error': maximum_score_error,
    'Default Predictions Match': True,
    'Optimized Predictions Match': True,
    'Thresholds Match': True,
    'Metric Reconstruction': 'Passed',
    'Row Alignment': 'Passed'
}])

model_reload_df.to_csv(
    TABLES_DIR / 'nb_model_reload_verification.csv',
    index=False
)

display(model_reload_df)

print("\nModel storage summary")
print("Model saved:", MODEL_PATH)
print("Validation prediction shape:",
      final_predictions_df.shape)
print("Artifacts fingerprinted:",
      len(nb_artifact_manifest_df))
print("Model reconstruction: Passed")
print("Metric reconstruction: Passed")
print("Official test dataset: Not used")
print("\nNaive Bayes model and predictions saved successfully.")

In [ ]:
# Register the completed Naive Bayes baseline experiment

registry_df = pd.read_csv(REGISTRY_PATH)

experiment_record = {
    'run_id': RUN_ID,
    'notebook': '03_classical_ml/01_naive_bayes',
    'model': MODEL_NAME,
    'feature_representation': FEATURE_REPRESENTATION,
    'loss_function': 'Not applicable',
    'sampling_strategy': 'representative',
    'training_samples': len(y_train),
    'seed': SEED,
    'max_length': None,
    'config_path': str(
        RUN_CONFIG_PATH.relative_to(PROJECT_ROOT)
    ),
    'model_path': str(
        MODEL_PATH.relative_to(PROJECT_ROOT)
    ),
    'results_path': str(
        RESULTS_DIR.relative_to(PROJECT_ROOT)
    ),
    'validation_macro_ap': optimized_metrics['Macro AP'],
    'validation_macro_f1': optimized_metrics['Macro F1'],
    'training_time_seconds': training_time,
    'status': 'Completed',
    'start_time': training_start,
    'end_time': training_end,
    'notes': (
        'Standard Multinomial NB with alpha=1.0 and empirical '
        'class priors. One-vs-Rest classification with 50000 '
        'word TF-IDF features. No class balancing. '
        'Macro F1 recorded using validation-optimized per-label '
        'thresholds. Model reload and metric reconstruction passed. '
        'Official test not used.'
    )
}

assert set(experiment_record.keys()) == set(registry_df.columns)

# Prevent duplicate experiment entries

registry_df = registry_df.loc[
    registry_df['run_id'] != RUN_ID
].copy()

new_record_df = pd.DataFrame(
    [experiment_record],
    columns=registry_df.columns
)

if registry_df.empty:
    registry_df = new_record_df
else:
    registry_df = pd.concat(
        [registry_df, new_record_df],
        ignore_index=True
    )

assert registry_df['run_id'].is_unique

registry_df.to_csv(
    REGISTRY_PATH,
    index=False
)

# Verify the saved experiment registration

saved_registry_df = pd.read_csv(REGISTRY_PATH)

assert saved_registry_df['run_id'].is_unique

saved_run = saved_registry_df.loc[
    saved_registry_df['run_id'] == RUN_ID
].iloc[0]

assert saved_run['status'] == 'Completed'

assert np.isclose(
    saved_run['validation_macro_ap'],
    optimized_metrics['Macro AP']
)

assert np.isclose(
    saved_run['validation_macro_f1'],
    optimized_metrics['Macro F1']
)

assert (PROJECT_ROOT / saved_run['model_path']).is_file()
assert (PROJECT_ROOT / saved_run['config_path']).is_file()

print("Shared experiment registry")

display(saved_registry_df[[
    'run_id',
    'model',
    'feature_representation',
    'training_samples',
    'validation_macro_ap',
    'validation_macro_f1',
    'training_time_seconds',
    'status'
]].round(6))

# Save a consolidated model performance record

nb_run_summary_df = pd.DataFrame([{
    'Run ID': RUN_ID,
    'Model': MODEL_NAME,
    'Feature Representation': FEATURE_REPRESENTATION,
    'Training Samples': len(y_train),
    'Validation Samples': len(y_val),
    'Features': X_train.shape[1],
    'Macro AP': optimized_metrics['Macro AP'],
    'Micro AP': optimized_metrics['Micro AP'],
    'Macro ROC-AUC': optimized_metrics['Macro ROC-AUC'],
    'Default Macro F1': default_metrics['Macro F1'],
    'Optimized Macro F1': optimized_metrics['Macro F1'],
    'Default Micro F1': default_metrics['Micro F1'],
    'Optimized Micro F1': optimized_metrics['Micro F1'],
    'Optimized Macro Precision': optimized_metrics['Macro Precision'],
    'Optimized Macro Recall': optimized_metrics['Macro Recall'],
    'Training Time (s)': training_time,
    'Validation Prediction Time (s)': prediction_time,
    'Model Size (MiB)': MODEL_PATH.stat().st_size / (1024 ** 2),
    'Threshold Strategy': 'Per-label validation F1',
    'Official Test Used': False,
    'Status': 'Completed'
}])

nb_run_summary_df.to_csv(
    TABLES_DIR / 'nb_run_summary.csv',
    index=False
)

print("\nRegistered experiment summary")
display(nb_run_summary_df.T)

print("\nTotal registered experiments:", len(saved_registry_df))
print("Duplicate run IDs: 0")
print("Experiment registry verification: Passed")
print("Experiment registered successfully.")

In [ ]:
# Verify all Naive Bayes experiment artifacts

required_tables = [
    'environment_versions.csv',
    'dataset_feature_verification.csv',
    'label_support_and_imbalance.csv',
    'evaluation_module_verification.csv',
    'expected_class_priors.csv',
    'nb_classifier_training_summary.csv',
    'nb_validation_score_summary.csv',
    'nb_default_overall_metrics.csv',
    'nb_default_per_label_metrics.csv',
    'nb_average_precision_reference.csv',
    'nb_all_negative_baseline_comparison.csv',
    'nb_optimized_thresholds.csv',
    'nb_threshold_comparison.csv',
    'nb_overall_threshold_comparison.csv',
    'nb_optimized_per_label_metrics.csv',
    'nb_optimized_overall_metrics.csv',
    'nb_model_artifact_manifest.csv',
    'nb_model_reload_verification.csv',
    'nb_run_summary.csv'
]

required_figures = [
    'nb_validation_score_distributions.png',
    'nb_threshold_performance_comparison.png',
    'nb_precision_recall_curves.png',
    'nb_confusion_matrix_comparison.png'
]

artifact_groups = {
    'Table': [
        TABLES_DIR / name for name in required_tables
    ],
    'Figure': [
        FIGURES_DIR / name for name in required_figures
    ],
    'Configuration': [
        RUN_CONFIG_PATH,
        THRESHOLD_PATH
    ],
    'Prediction': [
        PREDICTIONS_DIR / f'{RUN_ID}_validation_scores.parquet',
        FINAL_PREDICTIONS_PATH
    ],
    'Training Log': [
        LOGS_DIR / f'{RUN_ID}_training_log.json'
    ],
    'Model': [
        MODEL_PATH
    ],
    'Shared Dependency': [
        vectorizer_path,
        SRC_DIR / 'evaluation_utils.py'
    ]
}

final_verification_records = []

for category, paths in artifact_groups.items():

    for path in paths:

        available = path.is_file() and path.stat().st_size > 0

        final_verification_records.append({
            'Type': category,
            'Filename': path.name,
            'Path': str(path.relative_to(PROJECT_ROOT)),
            'Status': 'Available' if available else 'Missing'
        })

final_verification_df = pd.DataFrame(
    final_verification_records
)

final_verification_df.to_csv(
    TABLES_DIR / 'nb_final_artifact_verification.csv',
    index=False
)

assert final_verification_df['Status'].eq('Available').all()

# Recheck fingerprints of critical model artifacts

for _, record in nb_artifact_manifest_df.iterrows():

    path = PROJECT_ROOT / record['Path']

    assert calculate_sha256(path) == record['SHA256']

print("Final artifact verification")

display(
    final_verification_df
    .groupby(['Type', 'Status'])
    .size()
    .reset_index(name='Artifacts')
)

# Update the methodological decision log

DECISION_LOG_PATH = CONFIG_DIR / 'project_decisions.csv'

decisions_df = pd.read_csv(DECISION_LOG_PATH)

nb_decisions = [
    {
        'decision_id': 'D023',
        'component': 'Classical baseline configuration',
        'decision': (
            'Multinomial Naive Bayes with alpha=1.0 '
            'and empirical class priors'
        ),
        'justification': (
            'Standard reproducible probabilistic baseline using '
            'nonnegative word TF-IDF features and natural training '
            'prevalence. Establishes a reference before stronger '
            'classical and transformer experiments.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D024',
        'component': 'Multi-label classification strategy',
        'decision': (
            'One-vs-Rest for classical classifiers'
        ),
        'justification': (
            'Supports six independently predicted binary labels '
            'without imposing mutual exclusivity between toxicity '
            'categories.'
        ),
        'status': 'Selected'
    },
    {
        'decision_id': 'D025',
        'component': 'Evaluation framework',
        'decision': (
            'Shared evaluation and exact per-label threshold optimization'
        ),
        'justification': (
            'Consistent metric definitions across classifiers. '
            'Threshold optimization examines available validation '
            'score cutoffs and selects the highest threshold '
            'among tied maximum F1 values.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D026',
        'component': 'Naive Bayes experiment scope',
        'decision': 'Retain standard Naive Bayes baseline',
        'justification': (
            f"Validation Macro AP is "
            f"{optimized_metrics['Macro AP']:.6f}. "
            f"Threshold optimization improves Macro F1 from "
            f"{default_metrics['Macro F1']:.6f} to "
            f"{optimized_metrics['Macro F1']:.6f}. "
            'Further Naive Bayes tuning is deprioritized to '
            'focus on stronger classifiers and required '
            'transformer experiments.'
        ),
        'status': 'Implemented'
    }
]

for record in nb_decisions:

    mask = decisions_df['decision_id'].eq(
        record['decision_id']
    )

    if mask.any():

        for column, value in record.items():
            decisions_df.loc[mask, column] = value

    else:

        decisions_df = pd.concat(
            [decisions_df, pd.DataFrame([record])],
            ignore_index=True
        )

assert decisions_df['decision_id'].is_unique

decisions_df.to_csv(
    DECISION_LOG_PATH,
    index=False
)

print("\nNew methodological decisions")

display(
    decisions_df[
        decisions_df['decision_id'].isin(
            [record['decision_id'] for record in nb_decisions]
        )
    ][['decision_id', 'component', 'decision', 'status']]
)

# Save the final notebook summary

summary = f"""# Notebook 03.01 - Multinomial Naive Bayes

## Objective

Establish a standard probabilistic baseline for multi-label
toxic comment classification and investigate the effect
of validation-based per-label threshold optimization.

## Experimental Configuration

- Run ID: {RUN_ID}
- Training samples: {len(y_train):,}
- Validation samples: {len(y_val):,}
- Representation: Word TF-IDF
- Features: {X_train.shape[1]:,}
- Classification strategy: One-vs-Rest
- Binary classifiers: {len(LABELS)}
- Alpha: 1.0
- Empirical class priors: Enabled
- Class balancing: None
- Sampling: Representative
- Seed policy: {SEED}

## Validation Results

| Metric | Default | Optimized |
|---|---:|---:|
| Macro AP | {default_metrics['Macro AP']:.6f} | {optimized_metrics['Macro AP']:.6f} |
| Micro AP | {default_metrics['Micro AP']:.6f} | {optimized_metrics['Micro AP']:.6f} |
| Macro ROC-AUC | {default_metrics['Macro ROC-AUC']:.6f} | {optimized_metrics['Macro ROC-AUC']:.6f} |
| Macro Precision | {default_metrics['Macro Precision']:.6f} | {optimized_metrics['Macro Precision']:.6f} |
| Macro Recall | {default_metrics['Macro Recall']:.6f} | {optimized_metrics['Macro Recall']:.6f} |
| Macro F1 | {default_metrics['Macro F1']:.6f} | {optimized_metrics['Macro F1']:.6f} |
| Micro F1 | {default_metrics['Micro F1']:.6f} | {optimized_metrics['Micro F1']:.6f} |
| Hamming Loss | {default_metrics['Hamming Loss']:.6f} | {optimized_metrics['Hamming Loss']:.6f} |
| Exact Match Accuracy | {default_metrics['Exact Match Accuracy']:.6f} | {optimized_metrics['Exact Match Accuracy']:.6f} |

## Main Findings

- Per-label threshold optimization improves F1 for all six labels.
- Macro F1 improves by approximately 20.59 percentage points.
- Micro F1 improves by approximately 17.04 percentage points.
- Ranking metrics remain unchanged after threshold optimization.
- Default thresholds produce no positive predictions for
  threat and identity hate.
- Severe toxicity exhibits the largest absolute F1 improvement.
- Threat remains difficult to classify, with an optimized
  F1 of approximately 0.0884.
- Higher recall is accompanied by additional false positives.

## Computational Efficiency

- Model fitting time: {training_time:.6f} seconds.
- Validation prediction time: {prediction_time:.6f} seconds.
- Model storage: {MODEL_PATH.stat().st_size / (1024 ** 2):.4f} MiB.

Feature extraction time is reported separately in Notebook 03.00.

## Reproducibility

- All six fitted classifiers verified.
- Model reload verification passed.
- All {val_scores.size:,} validation scores reconstructed.
- Maximum absolute score reconstruction error: {maximum_score_error}.
- Saved thresholds and predictions verified.
- Dataset, feature and artifact fingerprints recorded.
- Experiment registered in the shared registry.

## Limitations

- Conditional feature independence is a simplifying assumption.
- TF-IDF values are fractional rather than raw word counts.
- Class imbalance and smoothing can affect score distributions.
- Probability calibration has not been established.
- Validation-optimized F1 is potentially optimistic because
  threshold selection and evaluation use the same partition.
- Alpha and class priors have not been optimized.
- Results do not establish final generalization performance.

## Evaluation Protocol

Macro Average Precision remains the primary model selection
metric. Official test data was not used for model training,
model selection or threshold optimization.

## Conclusion

The standard Naive Bayes baseline is complete and retained
for comparison with subsequent classical and transformer models.

## Next Stage

Notebook 03.02 - Logistic Regression.
"""

(RESULTS_DIR / 'summary.md').write_text(
    summary,
    encoding='utf-8'
)

assert (RESULTS_DIR / 'summary.md').stat().st_size > 0

print("\nNaive Bayes completion summary")
print("Analysis tables verified:", len(required_tables))
print("Research figures verified:", len(required_figures))
print("Total required artifacts:", len(final_verification_df))
print("Missing artifacts:",
      final_verification_df['Status'].eq('Missing').sum())
print("Total methodological decisions:", len(decisions_df))
print("Registered experiments:", len(saved_registry_df))
print("Notebook summary saved.")
print("Official test dataset: Not used")

print("\nNaive Bayes experiment completed successfully.")